# MARBO AI Cover — Google Colab

Ten notebook uruchamia **ACE-Step 1.5 na GPU Google Colab** i obsługuje kolejkę z folderu `Mój dysk/MARBO AI Cover`.

### Jak używać
1. W Colab wybierz **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU (np. T4)**.
2. Kliknij **Uruchom wszystko**.
3. Przy montowaniu Dysku Google zaakceptuj dostęp.
4. Ostatnia komórka działa jako kolejka. **Pozostaw ją uruchomioną**.
5. W programie Windows wybierz ten sam „Mój dysk”, MP3/WAV i kliknij **GENERUJ COVER AI**.

Darmowy Colab nie gwarantuje GPU ani czasu działania i może rozłączyć sesję.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, time, shutil, mimetypes, subprocess, sys

ROOT = Path("/content/drive/MyDrive/MARBO AI Cover")
QUEUE = ROOT / "Queue"
STATUS = ROOT / "Status"
OUTPUT = ROOT / "Output"
DONE = ROOT / "Done"

for p in (ROOT, QUEUE, STATUS, OUTPUT, DONE):
    p.mkdir(parents=True, exist_ok=True)

print("Folder roboczy:", ROOT)
print("Queue:", QUEUE)
print("Output:", OUTPUT)


In [ ]:
import os, subprocess, time, requests, torch
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK GPU")
if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab nie ma aktywnego GPU. Wybierz: Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU, potem uruchom notebook ponownie."
    )

ACE_DIR = Path("/content/ACE-Step-1.5")
API_BASE = "http://127.0.0.1:8001"
LOG_FILE = Path("/content/acestep-api.log")

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"], check=True)

if not shutil.which("uv"):
    subprocess.run(
        ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
        check=True,
    )
os.environ["PATH"] = os.environ.get("PATH", "") + ":/root/.local/bin"
UV = shutil.which("uv") or "/root/.local/bin/uv"

if not ACE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ACE-Step/ACE-Step-1.5.git", str(ACE_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=ACE_DIR, check=False)

print("Przygotowuję Python 3.12...")
subprocess.run([UV, "python", "install", "3.12"], check=True)
print("Instaluję zależności ACE-Step. Pierwszy raz może potrwać kilka-kilkanaście minut...")
subprocess.run([UV, "sync", "--python", "3.12"], cwd=ACE_DIR, check=True)

def api_healthy():
    try:
        return requests.get(API_BASE + "/health", timeout=3).ok
    except Exception:
        return False

if api_healthy():
    print("ACE-Step API już działa.")
else:
    env = os.environ.copy()
    env["ACESTEP_INIT_LLM"] = "false"
    env["ACESTEP_API_HOST"] = "127.0.0.1"
    env["ACESTEP_API_PORT"] = "8001"
    env["ACESTEP_CONFIG_PATH"] = "acestep-v15-turbo"

    log = open(LOG_FILE, "a", encoding="utf-8")
    api_proc = subprocess.Popen(
        [UV, "run", "acestep-api"],
        cwd=ACE_DIR,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

    print("Uruchamiam ACE-Step. Przy pierwszym starcie modele mogą się jeszcze pobierać...")
    deadline = time.time() + 20 * 60
    while time.time() < deadline:
        if api_healthy():
            print("ACE-Step API: GOTOWY.")
            break
        if api_proc.poll() is not None:
            print(LOG_FILE.read_text(encoding="utf-8", errors="ignore")[-6000:])
            raise RuntimeError("ACE-Step zakończył działanie. Ostatnie wpisy logu pokazano powyżej.")
        time.sleep(5)
    else:
        print(LOG_FILE.read_text(encoding="utf-8", errors="ignore")[-6000:])
        raise RuntimeError("ACE-Step nie uruchomił API w ciągu 20 minut.")


In [ ]:
import json, time, requests, mimetypes, shutil, traceback
from datetime import datetime
from pathlib import Path

API_BASE = "http://127.0.0.1:8001"

def atomic_json(path: Path, data: dict):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def set_status(job, status, message="", outputs=None):
    data = dict(job)
    data["status"] = status
    data["message"] = message
    data["updated_at"] = datetime.now().isoformat(timespec="seconds")
    if outputs is not None:
        data["outputs"] = outputs
    atomic_json(STATUS / f"{job['job_id']}.json", data)

def wait_for_synced_source(job, timeout=15*60):
    src = QUEUE / job["source_name"]
    target_size = int(job.get("source_size") or 0)
    end = time.time() + timeout
    last_size = -1
    stable = 0
    while time.time() < end:
        if src.exists():
            size = src.stat().st_size
            if target_size and size == target_size:
                return src
            if size == last_size and size > 0:
                stable += 1
                if stable >= 3 and not target_size:
                    return src
            else:
                stable = 0
            last_size = size
        time.sleep(5)
    raise TimeoutError("Plik audio nie zsynchronizował się z Dyskiem Google w wyznaczonym czasie.")

def submit_cover(job, src: Path):
    fields = {
        "prompt": job.get("prompt", ""),
        "task_type": "cover",
        "audio_cover_strength": str(job.get("audio_cover_strength", 0.68)),
        "audio_format": job.get("audio_format", "mp3"),
        "batch_size": str(job.get("batch_size", 2)),
        "model": job.get("model", "acestep-v15-turbo"),
        "inference_steps": str(job.get("inference_steps", 8)),
        "thinking": "false",
    }
    mime = mimetypes.guess_type(str(src))[0] or "application/octet-stream"
    with src.open("rb") as fh:
        files = {"src_audio": (src.name, fh, mime)}
        r = requests.post(API_BASE + "/release_task", data=fields, files=files, timeout=300)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 200:
        raise RuntimeError(payload.get("error") or f"release_task code={payload.get('code')}")
    return payload["data"]["task_id"]

def wait_task(task_id, job, timeout=60*60):
    end = time.time() + timeout
    while time.time() < end:
        r = requests.post(
            API_BASE + "/query_result",
            json={"task_id_list": [task_id]},
            timeout=30,
        )
        r.raise_for_status()
        payload = r.json()
        if payload.get("code") != 200:
            raise RuntimeError(payload.get("error") or "Błąd query_result")
        rows = payload.get("data") or []
        if rows:
            row = rows[0]
            status = int(row.get("status", 0))
            if status == 1:
                result = row.get("result") or "[]"
                return json.loads(result) if isinstance(result, str) else result
            if status == 2:
                raise RuntimeError("ACE-Step zgłosił błąd generowania.")
        set_status(job, "processing", "ACE-Step generuje aranżację...")
        time.sleep(8)
    raise TimeoutError("Generowanie trwało dłużej niż 60 minut.")

def download_results(job, results):
    fmt = job.get("audio_format", "mp3")
    saved = []
    for i, item in enumerate(results, start=1):
        file_url = item.get("file")
        if not file_url:
            continue
        if file_url.startswith("http://") or file_url.startswith("https://"):
            url = file_url
        else:
            url = API_BASE + file_url
        out = OUTPUT / f"{job['job_id']}_v{i}.{fmt}"
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with out.open("wb") as fh:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        fh.write(chunk)
        saved.append(out.name)
    if not saved:
        raise RuntimeError("Silnik zakończył zadanie, ale nie zwrócił pliku audio.")
    return saved

def archive_job(job, src, job_file):
    target = DONE / job["job_id"]
    target.mkdir(parents=True, exist_ok=True)
    try:
        if src.exists():
            shutil.move(str(src), str(target / src.name))
    except Exception:
        pass
    try:
        if job_file.exists():
            shutil.move(str(job_file), str(target / job_file.name))
    except Exception:
        pass

def process_job(job_file: Path):
    job = json.loads(job_file.read_text(encoding="utf-8"))
    job_id = job["job_id"]
    print(f"\n[{datetime.now():%H:%M:%S}] Zadanie {job_id}: {job.get('original_name')}")
    set_status(job, "processing", "Czekam na pełną synchronizację pliku audio...")
    src = wait_for_synced_source(job)
    try:
        set_status(job, "processing", "Wysyłam utwór do ACE-Step...")
        task_id = submit_cover(job, src)
        set_status(job, "processing", f"Generowanie AI. Task: {task_id}")
        results = wait_task(task_id, job)
        saved = download_results(job, results)
        set_status(job, "completed", "Gotowe.", saved)
        print("GOTOWE:", ", ".join(saved))
    except Exception as exc:
        msg = f"{type(exc).__name__}: {exc}"
        set_status(job, "failed", msg)
        print("BŁĄD:", msg)
        traceback.print_exc()
    finally:
        archive_job(job, src, job_file)

print("MARBO AI Cover — kolejka uruchomiona.")
print("Pozostaw tę komórkę aktywną. Program sprawdza Dysk Google co 5 sekund.")
print("Aby zatrzymać, użyj przycisku STOP przy komórce.\n")

while True:
    jobs = sorted(QUEUE.glob("*.json"))
    if not jobs:
        time.sleep(5)
        continue
    for job_file in jobs:
        try:
            process_job(job_file)
        except Exception as exc:
            print("Nie można odczytać zadania:", job_file.name, exc)
            time.sleep(3)
